# 01 · Pronóstico de compras observadas

**Ejercicio principal · ~1 hora · CRISP-DM completo.** En equipos, elijan al menos una categoría, periodo de agregación (mes/semana) u horizonte antes de ejecutar el modelado. Trabajaremos con pedidos entregados y contaremos filas de artículos; no son unidades ni demanda real: el dataset no informa inventario, quiebres ni compras no satisfechas.

Al final habrá una demo Gradio. En Colab la interfaz es temporal y no constituye un despliegue de producción.

## 1. Comprensión del negocio

**Pregunta:** ¿cuántas líneas de artículos de pedidos entregados podríamos observar en próximos periodos para una categoría? Esto puede apoyar una conversación de planeación, pero no mide ventas perdidas ni la demanda total. El equipo decide categoría, granularidad temporal y horizonte, y acuerda cómo usaría el pronóstico sin confundirlo con una garantía.

In [ ]:
from pathlib import Path

# Decisión del equipo: category=None usa todas; o escriban una categoría listada abajo.
CATEGORY = None
AGGREGATION = "M"  # "M" = mes, "W" = semana
HORIZON = 1  # periodos hacia adelante; elijan 1, 2 o 3
DATA_DIR = Path("data/processed")

## 2. Comprensión de los datos

Usaremos pedidos, líneas de pedido, productos y traducciones de categorías. El conversor y el [diccionario](docs/diccionario_datos.md) describen columnas y relaciones. Si falta un Parquet, descarguen los CSV oficiales y ejecuten `python scripts/prepare_data.py` desde la raíz del repositorio. No se crea un resultado de sustitución.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

required_files = [
    "olist_orders_dataset.parquet",
    "olist_order_items_dataset.parquet",
    "olist_products_dataset.parquet",
    "product_category_name_translation.parquet",
]
missing = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Faltan {missing} en {DATA_DIR}. Descargue Olist, coloque los CSV en "
        "data/raw/ y ejecute `python scripts/prepare_data.py` desde la raíz."
    )

orders = pd.read_parquet(DATA_DIR / required_files[0])
items = pd.read_parquet(DATA_DIR / required_files[1])
products = pd.read_parquet(DATA_DIR / required_files[2])
translation = pd.read_parquet(DATA_DIR / required_files[3])
print(f"Pedidos: {len(orders):,}; líneas de pedido: {len(items):,}")

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"], errors="coerce"
)
sales = (
    items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(translation, on="product_category_name", how="left")
    .merge(
        orders[["order_id", "order_status", "order_purchase_timestamp"]],
        on="order_id",
        how="inner",
    )
)
sales["category"] = sales["product_category_name_english"].fillna(
    sales["product_category_name"]
).fillna("sin_categoria")
sales = sales.loc[
    sales["order_status"].eq("delivered")
    & sales["order_purchase_timestamp"].notna()
].copy()
print("Categorías disponibles (más frecuentes):")
display(sales["category"].value_counts().head(12).rename("líneas observadas"))

## 3. Preparación de los datos

Contaremos registros de `order_items` por categoría y periodo, no una columna `quantity` (no está en el esquema). Los periodos sin registros se completan con cero. Las tres últimas observaciones previas forman las variables de rezago. El conjunto de prueba se reserva al final del calendario, nunca se barajan fechas.

In [ ]:
if AGGREGATION not in {"M", "W"}:
    raise ValueError("AGGREGATION debe ser 'M' (mes) o 'W' (semana).")
if HORIZON not in {1, 2, 3}:
    raise ValueError("HORIZON debe ser 1, 2 o 3 periodos.")
if CATEGORY is not None:
    if CATEGORY not in set(sales["category"]):
        raise ValueError("CATEGORY no coincide con las categorías listadas arriba.")
    sales = sales.loc[sales["category"].eq(CATEGORY)].copy()

sales["period"] = sales["order_purchase_timestamp"].dt.to_period(
    "M" if AGGREGATION == "M" else "W-SUN"
).dt.start_time
counts = sales.groupby(["category", "period"], observed=True).size()
series_parts = []
for category, values in counts.groupby(level="category"):
    values = values.droplevel("category").sort_index()
    calendar = pd.date_range(values.index.min(), values.index.max(), freq="MS" if AGGREGATION == "M" else "W-MON")
    complete = values.reindex(calendar, fill_value=0)
    series_parts.append(pd.DataFrame({"category": category, "date": calendar, "observed_lines": complete.to_numpy()}))
panel = pd.concat(series_parts, ignore_index=True)
panel = panel.sort_values(["category", "date"])
for lag in (1, 2, 3):
    panel[f"lag_{lag}"] = panel.groupby("category")["observed_lines"].shift(lag)
panel["target"] = panel.groupby("category")["observed_lines"].shift(-HORIZON)
panel["target_date"] = panel.groupby("category")["date"].shift(-HORIZON)
panel["target_month"] = panel["target_date"].dt.month
panel = panel.dropna(subset=["lag_1", "lag_2", "lag_3", "target"]).copy()
panel["horizon"] = HORIZON
panel["month_sin"] = np.sin(2 * np.pi * panel["target_month"] / 12)
panel["month_cos"] = np.cos(2 * np.pi * panel["target_month"] / 12)
print(f"Filas de modelado: {len(panel):,}; horizonte: {HORIZON} periodo(s)")

## 4. Modelado

La referencia ingenua repite el último periodo conocido (`lag_1`). La comparamos con un Random Forest que aprende patrones no lineales y codifica categoría sin ordenarla artificialmente. El horizonte seleccionado se aplica a todos los ejemplos supervisados.

In [ ]:
dates = np.sort(panel["target_date"].unique())
if len(dates) < 3:
    raise ValueError("No hay suficientes periodos para separar entrenamiento y prueba temporal.")
split_date = dates[max(1, int(len(dates) * 0.8)) - 1]
train = panel.loc[panel["target_date"] <= split_date].copy()
test = panel.loc[panel["target_date"] > split_date].copy()
if train.empty or test.empty:
    raise ValueError("La partición temporal quedó vacía; revise el rango de fechas.")

feature_columns = ["category", "lag_1", "lag_2", "lag_3", "horizon", "month_sin", "month_cos"]
preprocess = ColumnTransformer(
    [("category", OneHotEncoder(handle_unknown="ignore"), ["category"])],
    remainder="passthrough",
)
model = make_pipeline(
    preprocess,
    RandomForestRegressor(n_estimators=120, min_samples_leaf=2, random_state=42, n_jobs=-1),
)
model.fit(train[feature_columns], train["target"])
test["baseline_prediction"] = test["lag_1"]
test["model_prediction"] = model.predict(test[feature_columns])
print(f"Último periodo de entrenamiento: {pd.Timestamp(split_date).date()}")
print(f"Primera fecha objetivo de prueba: {test['target_date'].min().date()}")

## 5. Evaluación

MAE es el error absoluto medio en líneas observadas por periodo; menor es mejor. La partición mantiene el orden temporal. Un buen MAE en este histórico no prueba que el modelo generalice a cambios de surtido, disponibilidad o comportamiento.

In [ ]:
scores = pd.DataFrame(
    {
        "modelo": ["Referencia: último periodo", "Random Forest"],
        "MAE": [
            mean_absolute_error(test["target"], test["baseline_prediction"]),
            mean_absolute_error(test["target"], test["model_prediction"]),
        ],
    }
)
display(scores.round(2))

In [ ]:
plot_category = CATEGORY or test["category"].value_counts().index[0]
plot_data = test.loc[test["category"].eq(plot_category)].sort_values("target_date")
if plot_data.empty:
    raise ValueError("No hay ejemplos de prueba para la categoría elegida.")
plt.figure(figsize=(10, 4))
plt.plot(plot_data["target_date"], plot_data["target"], marker="o", label="Observado")
plt.plot(plot_data["target_date"], plot_data["model_prediction"], marker="o", label="Pronosticado")
plt.title(f"Líneas de artículo observadas vs. pronosticadas · {plot_category}")
plt.xlabel("Periodo objetivo")
plt.ylabel("Líneas en pedidos entregados")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 6. Despliegue (demostración temporal)

Prueben escenarios cambiando categoría, rezagos y mes objetivo. El modelo fue ajustado para el horizonte fijado arriba. Esta pequeña interfaz de Gradio solo ilustra una entrada/salida; en Colab es temporal, no persiste al cerrar la sesión y no es un despliegue de producción. No incluye inventario ni pronostica demanda no observada.

In [ ]:
import gradio as gr

categories_for_demo = sorted(panel["category"].unique().tolist())

def forecast_demo(category, lag_1, lag_2, lag_3, target_month):
    row = pd.DataFrame(
        {
            "category": [category],
            "lag_1": [lag_1],
            "lag_2": [lag_2],
            "lag_3": [lag_3],
            "horizon": [HORIZON],
            "month_sin": [np.sin(2 * np.pi * target_month / 12)],
            "month_cos": [np.cos(2 * np.pi * target_month / 12)],
        }
    )
    estimate = max(0.0, float(model.predict(row[feature_columns])[0]))
    return f"Estimación: {estimate:.1f} líneas de artículo en el periodo elegido."

demo = gr.Interface(
    fn=forecast_demo,
    inputs=[
        gr.Dropdown(categories_for_demo, value=plot_category, label="Categoría"),
        gr.Number(value=10, minimum=0, label="Líneas del periodo anterior"),
        gr.Number(value=10, minimum=0, label="Líneas hace 2 periodos"),
        gr.Number(value=10, minimum=0, label="Líneas hace 3 periodos"),
        gr.Slider(1, 12, value=1, step=1, label="Mes del periodo objetivo"),
    ],
    outputs=gr.Textbox(label="Pronóstico exploratorio"),
    title="Pronóstico de líneas observadas",
    description=f"Horizonte fijado: {HORIZON} periodo(s). No representa demanda real ni producción.",
)
demo.launch(share=False, inline=True, prevent_thread_lock=True)